# Asset Location ETL & Data Cleaning
### AMS Legacy Migration to Smart Microsoft SQL Server (`locations` table)

---

## 1. Overview & Business Context
This notebook performs data cleaning, normalization, relational hierarchy derivation, deduplication, and anomaly resolution on the legacy AMS (Asset Management System) location dataset (`data/ams/vw_assets_location_202609302141.csv`).

The goal is to prepare clean, verified data ready for direct import into the Microsoft SQL Server **`locations`** master table for the **Smart** application.

### Key Business Transformations:
1. **Self-Referencing Hierarchy Resolution (`parent_id`)**:
   - SMART uses a circular self-referencing key (`parent_id`) to designate location levels.
   - **Level 1 (Top / Building, `cap == 'B'`)**: `parent_id = NULL`
   - **Level 2 (Floor, `cap == 'F'`)**: `parent_id = Building ID` (resolved via `gedung` code)
   - **Level 3 (Room, `cap == 'R'`)**: `parent_id = Floor ID` (resolved via `lantai` code)
2. **Preservation of Legacy Primary Key (`id`)**:
   - The primary key `id` directly preserves original AMS IDs for transparent legacy cross-referencing and backward compatibility.
3. **Text Standardization & Typo Resolutions**:
   - Whitespace trimming (`Lift `, `Musholla `, `Toilet `).
   - Capitalization & Title Case alignment (`Area Tangga Darurat`, `Halaman Timur`, `Pos Security`, `Area Tangga Utama`).
   - Typographical spelling corrections (`Smoking Room`, `Area Sekretaris`, `Koridor Luar`, `Koridor Dalam`, `Departemen Process`, `Ruang Instrument, Mechanical, Process`).
   - Acronym preservation (`ICT`, `HRM`, `CFS`, `PPC`, `3DD`, `GM`, `DM`, `HSE`, `IFS`, `SDD`, `CSS`, `RETO`, `PHR`, `PLN`, `STP`, `AC AHU`, `VIP`, `SDQA`).
4. **Duplicate Room Consolidation**:
   - AMS ID 207 and ID 217 are identical duplicate entries for `Ruang Meeting Ratu Ratih` on Top Floor (ID 179) with code `115002`.
   - Primary **ID 207** is retained in the production master.
   - Secondary **ID 217** is relegated to `relegated_locations.csv`, and both IDs map to SMART ID 207 in `location_ams_mapping.csv`.
5. **Deleted Location Status Alignment (`is_active = 0`)**:
   - AMS records with `deletion = 1.0` (ID 176: `Process Department` and ID 192: `Ruang Electrical`) are imported with `is_active = 0` to preserve historical referential integrity while marking them inactive.
   - All other locations are set to `is_active = 1`.
6. **Relegation of Empty Records**:
   - AMS ID 219 (row with all NaN/0 values) is excluded from production and tracked in `relegated_locations.csv`.
7. **Placeholder Locations**:
   - ID 77 (`NODATA`) and IDs 213–215 (`Others Building`, `Others Floor`, `Others Room`) are retained as fallback targets for non-office site assets.
8. **Outputs**:
   - Staging: `data/smart/staging/location/` (`locations.csv`, `location_ams_mapping.csv`, `relegated_locations.csv`).
   - Production: `data/smart/final/location/` (`locations.csv`, `locations.sql`).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------
# Dynamic Path Configurations
# Robustly resolves data directory regardless of whether
# current working directory is /home/jovyan, /home/jovyan/work,
# or the repository root.
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path("data"),                                  # Running from project root
        Path("/home/jovyan/data"),                      # Running inside docker container
        Path("../data"),                               # Running from notebooks/ directory
        Path("/home/ran/Devs/ams-smart-migration/data") # Absolute host path
    ]
    for p in candidates:
        if p.exists() and (p / "ams").exists():
            return p.resolve()
    return Path("data").resolve()

DATA_DIR = get_data_dir()
RAW_DATA_PATH = DATA_DIR / "ams" / "vw_assets_location_202609302141.csv"
STAGING_DIR = DATA_DIR / "smart" / "staging" / "location"
FINAL_DIR = DATA_DIR / "smart" / "final" / "location"

STAGING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory:       {DATA_DIR}")
print(f"Raw AMS Location CSV: {RAW_DATA_PATH} (exists: {RAW_DATA_PATH.exists()})")
print(f"Staging directory:    {STAGING_DIR}")
print(f"Final directory:      {FINAL_DIR}")


Data directory:       /home/jovyan/data
Raw AMS Location CSV: /home/jovyan/data/ams/vw_assets_location_202609302141.csv (exists: True)
Staging directory:    /home/jovyan/data/smart/staging/location
Final directory:      /home/jovyan/data/smart/final/location


In [2]:
# Load raw location dataset, treating literal string 'NULL', 'null', and blanks as NaN
df_raw = pd.read_csv(
    RAW_DATA_PATH,
    na_values=["NULL", "null", "NaN", "nan", ""],
    keep_default_na=True
)

print(f"Loaded {len(df_raw)} total rows from AMS location dataset.")
print("Columns:", df_raw.columns.tolist())
display(df_raw.head(10))


Loaded 173 total rows from AMS location dataset.
Columns: ['id', 'code', 'gedung', 'lantai', 'lantai_name', 'gedung_name', 'location', 'location_name', 'department_related', 'description', 'create_date', 'cap', 'meeting_available', 'create_owner', 'loan_price', 'update_date', 'update_owner', 'deletion', 'deletion_date', 'deletion_owner']


,id,code,gedung,lantai,lantai_name,gedung_name,location,location_name,department_related,description,create_date,cap,meeting_available,create_owner,loan_price,update_date,update_owner,deletion,deletion_date,deletion_owner
0,36,100000.0,100000,100000,NaN,NaN,Graha RE 1,Graha RE 1,NaN,Graha RE 1,2016-12-08 15:02:54.283,B,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
1,37,110000.0,110000,110000,NaN,NaN,Graha RE 2,Graha RE 2,NaN,Graha RE 2,2016-12-08 15:03:03.060,B,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
2,38,101000.0,100000,101000,NaN,NaN,Lantai Mezzanine,Lantai Mezzanine,NaN,Lantai Dasar,2016-12-08 15:12:03.347,F,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
3,39,102000.0,100000,102000,NaN,NaN,Lantai 1,Lantai 1,NaN,Lantai 1,2016-12-08 15:12:10.520,F,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
4,41,103000.0,100000,103000,NaN,NaN,Lantai 2,Lantai 2,NaN,Lantai 2,2016-12-08 15:13:05.080,F,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
5,42,104000.0,100000,104000,NaN,NaN,Lantai 3,Lantai 3,NaN,Lantai 3,2016-12-08 15:13:12.610,F,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
6,44,111000.0,110000,111000,NaN,NaN,Basement,Basement,NaN,Basement,2016-12-08 15:13:51.240,F,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
7,45,112000.0,110000,112000,NaN,NaN,Mezzanine,Mezzanine,NaN,Mezzanine,2016-12-08 15:13:59.240,F,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
8,46,113000.0,110000,113000,NaN,NaN,Lantai 1,Lantai 1,NaN,Lantai 1,2016-12-08 15:14:06.450,F,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN
9,47,114000.0,110000,114000,NaN,NaN,Lantai 2,Lantai 2,NaN,Lantai 2,2016-12-08 15:14:59.713,F,0.0,302716,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Exploratory Analysis & Anomaly Detection
In this section, we analyze the location hierarchy hints (`cap`), detect unlinked records, identify duplicate names/codes, and inspect deleted rows.


In [3]:
# 1. Check cap value distribution
print("=== Hierarchy Level Hint (cap) Value Counts ===")
print(df_raw["cap"].value_counts(dropna=False))

# 2. Identify empty/unlinked rows
print("\n=== Unlinked / Invalid cap Rows ===")
invalid_cap = df_raw[~df_raw["cap"].isin(["B", "F", "R"])]
print(invalid_cap[["id", "code", "gedung", "lantai", "location", "cap"]])

# 3. Check deleted records in AMS
print("\n=== Deleted Records in AMS (deletion not null) ===")
deleted_records = df_raw[df_raw["deletion"].notnull()]
print(deleted_records[["id", "code", "gedung", "lantai", "location", "cap", "deletion", "deletion_date"]])

# 4. Check duplicate codes
print("\n=== Duplicate Codes in AMS ===")
dup_codes = df_raw[df_raw["code"].notnull() & df_raw.duplicated(subset=["code"], keep=False)]
print(dup_codes[["id", "code", "location", "cap", "create_date"]])

# 5. Check duplicate room ID 207 vs ID 217
print("\n=== Duplicate Room Check: Ruang Meeting Ratu Ratih ===")
print(df_raw[df_raw["id"].isin([207, 217])][["id", "code", "gedung", "lantai", "location", "create_date"]])


=== Hierarchy Level Hint (cap) Value Counts ===
cap
R      157
F       11
B        4
NaN      1
Name: count, dtype: int64

=== Unlinked / Invalid cap Rows ===
      id  code  gedung  lantai location  cap
145  219   NaN       0       0      NaN  NaN

=== Deleted Records in AMS (deletion not null) ===
      id      code  gedung  lantai            location cap  deletion  \
104  176  105009.0  100000  105000  Process Department   R       1.0   
120  192  112009.0  110000  112000    Ruang Electrical   R       1.0   

               deletion_date  
104                      NaN  
120  2020-10-19 00:00:00.000  

=== Duplicate Codes in AMS ===
      id      code                  location cap              create_date
135  207  115002.0  Ruang Meeting Ratu Ratih   R  2017-01-19 10:20:13.433
136  208  115003.0               Toilet Pria   R  2017-01-19 10:21:47.143
143  217  115002.0  Ruang Meeting Ratu Ratih   R  2019-11-18 09:11:15.600
161  235  115003.0        Ruang Kelas Recare   R  2023-06-13 

## 4. Master Location Configuration, Cleaning Rules, & Hierarchy Derivation

We now apply:
1. **Parent ID Derivation**: Building $\rightarrow$ Floor $\rightarrow$ Room mapping using 6-digit AMS codes.
2. **Text Cleaning & Typo Standardizations**: Applying curated dictionary replacements and trimming.
3. **Duplicate Consolidation**: Merging AMS ID 217 into ID 207.
4. **Deleted Status Assignment**: Setting `is_active = False` for IDs 176 and 192.
5. **Relegation Tracking**: Documenting relegated entries (IDs 217 & 219) for audit trail.
6. **DataFrames Construction**: Creating Staging, Mapping, and Final Production DataFrames.


In [4]:
# ---------------------------------------------------------
# 1. Curated Name Standardizations & Typo Corrections
# ---------------------------------------------------------
NAME_REPLACEMENTS = {
    # Capitalization & Title Case standardizations
    "Area tangga darurat": "Area Tangga Darurat",
    "Area tangga utama": "Area Tangga Utama",
    "Halaman timur": "Halaman Timur",
    "Pos security": "Pos Security",
    
    # Indonesian spelling & typo fixes
    "Smooking Room": "Smoking Room",
    "Area Sekertaris": "Area Sekretaris",
    "Kooridor Luar": "Koridor Luar",
    "Kooridor Dalam": "Koridor Dalam",
    "Departemen Proccess": "Departemen Process",
    "Ruang Instument, Mechanical, Proccess": "Ruang Instrument, Mechanical, Process",
}

def clean_location_name(val):
    if pd.isna(val) or val is None:
        return val
    cleaned = str(val).strip()
    return NAME_REPLACEMENTS.get(cleaned, cleaned)

# ---------------------------------------------------------
# 2. Build Code to ID Lookup for Valid AMS Records
# ---------------------------------------------------------
code_to_id = {}
for _, row in df_raw.iterrows():
    if pd.notnull(row["code"]) and pd.notnull(row["id"]):
        code_to_id[int(row["code"])] = int(row["id"])

# ---------------------------------------------------------
# 3. Hierarchy Level & Parent ID Resolution
# ---------------------------------------------------------
def resolve_parent_and_level(row):
    cap = row["cap"]
    if cap == "B":
        return None, 1 # Level 1: Building (Top level, parent_id is NULL)
    elif cap == "F":
        g_code = int(row["gedung"]) if pd.notnull(row["gedung"]) else None
        return code_to_id.get(g_code), 2 # Level 2: Floor
    elif cap == "R":
        l_code = int(row["lantai"]) if pd.notnull(row["lantai"]) else None
        return code_to_id.get(l_code), 3 # Level 3: Room
    return None, None

# ---------------------------------------------------------
# 4. Construct Staging, Mapping, and Relegated Datasets
# ---------------------------------------------------------
staging_rows = []
mapping_rows = []
relegated_rows = []

# Excluded duplicate ID 217 will be merged into ID 207
MERGED_DUPLICATES = {
    217: 207  # AMS 217 (Ruang Meeting Ratu Ratih) -> SMART 207
}

for _, row in df_raw.iterrows():
    raw_id = int(row["id"])
    raw_code = int(row["code"]) if pd.notnull(row["code"]) else None
    raw_name = row["location"]
    
    # Case A: Empty unmapped record (ID 219)
    if pd.isna(raw_name) or pd.isna(row["cap"]):
        relegated_rows.append({
            "ams_id": raw_id,
            "ams_code": raw_code,
            "ams_location_name": raw_name,
            "reason": "Empty record with NaN location name and cap",
            "created_at": row["create_date"]
        })
        mapping_rows.append({
            "ams_id": raw_id,
            "ams_code": raw_code,
            "ams_location_name": raw_name,
            "smart_location_id": None,
            "smart_name": None,
            "level": None,
            "notes": "Empty unmapped record"
        })
        continue

    parent_id, level = resolve_parent_and_level(row)
    cleaned_name = clean_location_name(raw_name)
    
    # Format timestamps to standard 'YYYY-MM-DD HH:MM:SS'
    created_dt = pd.to_datetime(row["create_date"]).strftime("%Y-%m-%d %H:%M:%S")
    updated_dt = created_dt
    
    # Active status: False (0) for confirmed deleted records 176 & 192, True (1) for others
    is_active = 0 if pd.notnull(row["deletion"]) and float(row["deletion"]) == 1.0 else 1
    
    # Related department: cast float to int if present
    rel_dept = int(row["department_related"]) if pd.notnull(row["department_related"]) else None
    
    # Case B: Secondary duplicate record (ID 217)
    if raw_id in MERGED_DUPLICATES:
        primary_id = MERGED_DUPLICATES[raw_id]
        relegated_rows.append({
            "ams_id": raw_id,
            "ams_code": raw_code,
            "ams_location_name": raw_name,
            "reason": f"Duplicate room of primary ID {primary_id}",
            "created_at": created_dt
        })
        mapping_rows.append({
            "ams_id": raw_id,
            "ams_code": raw_code,
            "ams_location_name": raw_name,
            "smart_location_id": primary_id,
            "smart_name": cleaned_name,
            "level": level,
            "notes": f"Merged duplicate of ID {primary_id}"
        })
        # Append legacy AMS ID to the primary record in staging
        for st_row in staging_rows:
            if st_row["id"] == primary_id:
                st_row["ams_id"].append(raw_id)
        continue

    # Case C: Primary Master Record
    ams_ids = [raw_id]
    staging_rows.append({
        "id": raw_id,
        "name": cleaned_name,
        "parent_id": parent_id,
        "related_departement": rel_dept,
        "is_active": is_active,
        "level": level,
        "cap": row["cap"],
        "ams_id": ams_ids,
        "ams_code": raw_code,
        "ams_gedung": int(row["gedung"]) if pd.notnull(row["gedung"]) else None,
        "ams_lantai": int(row["lantai"]) if pd.notnull(row["lantai"]) else None,
        "original_name": raw_name,
        "meeting_available": int(row["meeting_available"]) if pd.notnull(row["meeting_available"]) else 0,
        "created_at": created_dt,
        "updated_at": updated_dt
    })
    
    mapping_rows.append({
        "ams_id": raw_id,
        "ams_code": raw_code,
        "ams_location_name": raw_name,
        "smart_location_id": raw_id,
        "smart_name": cleaned_name,
        "level": level,
        "notes": "Primary master record"
    })

df_staging = pd.DataFrame(staging_rows)
df_mapping = pd.DataFrame(mapping_rows)
df_relegated = pd.DataFrame(relegated_rows)

# ---------------------------------------------------------
# 5. Build Final Production DataFrame (Topological Ordering)
# ---------------------------------------------------------
# Sort topologically by level (Level 1 Buildings first, Level 2 Floors second, Level 3 Rooms third)
# so parent records are inserted before child records in MS SQL Server.
df_final = df_staging[
    ["id", "name", "parent_id", "related_departement", "is_active", "created_at", "updated_at", "level"]
].sort_values(["level", "id"]).copy()

# Drop temporary 'level' column for strict 7-column schema
df_final = df_final[["id", "name", "parent_id", "related_departement", "is_active", "created_at", "updated_at"]].reset_index(drop=True)

print(f"Staging master records:    {len(df_staging)}")
print(f"Final production records:  {len(df_final)}")
print(f"Mapping entries (100%):    {len(df_mapping)} / {len(df_raw)}")
print(f"Relegated audit entries:   {len(df_relegated)}")
display(df_final.head(10))


Staging master records:    171
Final production records:  171
Mapping entries (100%):    173 / 173
Relegated audit entries:   2


,id,name,parent_id,related_departement,is_active,created_at,updated_at
0,36,Graha RE 1,NaN,NaN,1,2016-12-08 15:02:54,2016-12-08 15:02:54
1,37,Graha RE 2,NaN,NaN,1,2016-12-08 15:03:03,2016-12-08 15:03:03
2,77,NODATA,NaN,NaN,1,2017-01-04 15:18:18,2017-01-04 15:18:18
3,213,Others Building,NaN,NaN,1,2017-02-20 11:07:31,2017-02-20 11:07:31
4,38,Lantai Mezzanine,36.0,NaN,1,2016-12-08 15:12:03,2016-12-08 15:12:03
5,39,Lantai 1,36.0,NaN,1,2016-12-08 15:12:10,2016-12-08 15:12:10
6,41,Lantai 2,36.0,NaN,1,2016-12-08 15:13:05,2016-12-08 15:13:05
7,42,Lantai 3,36.0,NaN,1,2016-12-08 15:13:12,2016-12-08 15:13:12
8,44,Basement,37.0,NaN,1,2016-12-08 15:13:51,2016-12-08 15:13:51
9,45,Mezzanine,37.0,NaN,1,2016-12-08 15:13:59,2016-12-08 15:13:59


## 5. Data Validation & Quality Checks
We verify the integrity, constraints, DAG structure, and referential validity of the final dataset.


In [5]:
# 1. Check name length constraint (VARCHAR(255))
max_name_len = df_final["name"].str.len().max()
assert max_name_len <= 255, f"Error: Maximum location name length exceeds 255 characters ({max_name_len})"

# 2. Check uniqueness of (parent_id, name)
dups_per_parent = df_final[df_final.duplicated(subset=["parent_id", "name"], keep=False)]
assert dups_per_parent.empty, f"Error: Duplicate location names under the same parent detected: {dups_per_parent[['id', 'name', 'parent_id']].to_dict(orient='records')}"

# 3. Check Primary Key ID uniqueness
assert df_final["id"].duplicated().sum() == 0, "Error: Duplicate IDs found in final dataset"

# 4. Check Referential Integrity (all parent_ids must exist in id list)
valid_ids = set(df_final["id"])
orphans = df_final[df_final["parent_id"].notnull() & ~df_final["parent_id"].isin(valid_ids)]
assert orphans.empty, f"Error: Orphan records with nonexistent parent_id found: {orphans[['id', 'name', 'parent_id']].to_dict(orient='records')}"

# 5. Check DAG Structure & Zero Cycles
parent_map = {row["id"]: (row["parent_id"] if pd.notnull(row["parent_id"]) else None) for _, row in df_final.iterrows()}
for node_id in parent_map:
    visited = set()
    curr = node_id
    depth = 0
    while curr is not None:
        assert curr not in visited, f"Error: Circular reference cycle detected involving location ID {curr}"
        visited.add(curr)
        curr = parent_map.get(curr)
        depth += 1
        assert depth <= 5, f"Error: Location hierarchy depth exceeded expected tree depth at ID {node_id}"

# 6. Check for unexpected NULL values in required fields
null_counts = df_final[["id", "name", "is_active", "created_at", "updated_at"]].isna().sum()
assert null_counts.sum() == 0, f"Error: Found NULL values in required columns: {null_counts.to_dict()}"

# 7. Check that all 173 original AMS rows are completely mapped
assert len(df_mapping) == len(df_raw), f"Error: Mapping table length ({len(df_mapping)}) does not match raw AMS rows ({len(df_raw)})"

# 8. Check Topological Insertion Order (Parent must appear BEFORE child in df_final)
seen_ids = set()
for _, row in df_final.iterrows():
    p_id = row["parent_id"]
    if pd.notnull(p_id):
        assert int(p_id) in seen_ids, f"Error: Child location ID {row['id']} appears before parent ID {int(p_id)}"
    seen_ids.add(int(row["id"]))

print("All 8 data validation, integrity, and DAG topological assertions PASSED successfully!")


All 8 data validation, integrity, and DAG topological assertions PASSED successfully!


## 6. Export Outputs
We export staging files and final production files in two separate cells:
1. **Cell 10**: Staging files (`data/smart/staging/location/`)
2. **Cell 11**: Final production files (`data/smart/final/location/`)


In [6]:
# ---------------------------------------------------------
# Export Staging Outputs to data/smart/staging/location/
# ---------------------------------------------------------
staging_loc_path = STAGING_DIR / "locations.csv"
staging_map_path = STAGING_DIR / "location_ams_mapping.csv"
staging_relegated_path = STAGING_DIR / "relegated_locations.csv"

# 1. Staging locations with legacy codes & audit columns
df_staging.to_csv(staging_loc_path, index=False)

# 2. 1-to-1 Relational lookup mapping table
df_mapping.to_csv(staging_map_path, index=False)

# 3. Relegated audit table
df_relegated.to_csv(staging_relegated_path, index=False)

print(f"Staging files successfully exported to {STAGING_DIR.resolve()}:")
print(f" 1. {staging_loc_path.name} ({len(df_staging)} rows)")
print(f" 2. {staging_map_path.name} ({len(df_mapping)} rows)")
print(f" 3. {staging_relegated_path.name} ({len(df_relegated)} rows)")


Staging files successfully exported to /home/jovyan/data/smart/staging/location:
 1. locations.csv (171 rows)
 2. location_ams_mapping.csv (173 rows)
 3. relegated_locations.csv (2 rows)


In [7]:
# ---------------------------------------------------------
# Export Production Outputs to data/smart/final/location/
# ---------------------------------------------------------
final_csv_path = FINAL_DIR / "locations.csv"
final_sql_path = FINAL_DIR / "locations.sql"

# 1. Target CSV for SQL Server Import (Strict 7 Columns)
df_final.to_csv(final_csv_path, index=False)

# 2. Generate SQL Server INSERT Script (Topologically Ordered)
sql_statements = [
    "-- ========================================================",
    "-- Master Locations Data Migration Script for MS SQL Server",
    f"-- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "-- Target Table: locations",
    "-- ========================================================",
    "SET IDENTITY_INSERT [locations] ON;",
    "GO",
    ""
]

for _, row in df_final.iterrows():
    name_escaped = str(row['name']).replace("'", "''")
    parent_val = "NULL" if pd.isna(row['parent_id']) or row['parent_id'] is None else str(int(row['parent_id']))
    dept_val = "NULL" if pd.isna(row['related_departement']) or row['related_departement'] is None else str(int(row['related_departement']))
    active_val = "1" if int(row['is_active']) == 1 else "0"
    
    stmt = (
        f"INSERT INTO [locations] ([id], [name], [parent_id], [related_departement], [is_active], [created_at], [updated_at]) "
        f"VALUES ({int(row['id'])}, N'{name_escaped}', {parent_val}, {dept_val}, {active_val}, '{row['created_at']}', '{row['updated_at']}');"
    )
    sql_statements.append(stmt)

sql_statements.extend([
    "",
    "GO",
    "SET IDENTITY_INSERT [locations] OFF;",
    "GO"
])

with open(final_sql_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements))

print(f"Production files successfully exported to {FINAL_DIR.resolve()}:")
print(f" 1. {final_csv_path.name} ({len(df_final)} rows)")
print(f" 2. {final_sql_path.name} ({len(df_final)} INSERT statements)")


Production files successfully exported to /home/jovyan/data/smart/final/location:
 1. locations.csv (171 rows)
 2. locations.sql (171 INSERT statements)
